# True and approximate error

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/numerical_error/true_and_approximate_error.ipynb)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## True Absolute Error

The **True Absolute Error** is the absolute difference between the true value and an approximation.

$$
E_t = |\text{True} - \text{Approximate}|
$$

This metric is only useful when the true value is known.

### Example: What is the derivative of $2\sin(x)$ at $x=0$?

Recall the definition of a derivative:

$$
\frac{\mathrm{d}}{\mathrm{d}x} f(x) = \lim_{\Delta x \to 0} \frac{f(x+\Delta x) - f(x)}{\Delta x}
$$

The true value of the derivative of $2\sin(x)$ is $2\cos(x)$.

In [ ]:
value_true = 2*np.cos(0)
print(f"The true value is: {value_true}")

### Approximating the derivative

We can define a function to approximate the derivative using a finite $\Delta x$, and use it with $\Delta x = 0.1$, to find the approximate value and the true absolute error:

In [ ]:
def approx_derivative(delta_x):
    return (2*np.sin(0 + delta_x) - 2*np.sin(0)) / delta_x

value_approx = approx_derivative(delta_x=0.1)
print(f"The approximate value is: {value_approx}")

E_t = np.abs(value_true - value_approx)
print(f"The true absolute error is: {E_t}")

## True Relative Error

Often, the absolute error is not as useful as the **True Relative Error**, because it does not account for the magnitude of the value. For example, a 1-meter error in GPS is insignificant for a long road trip but critical for a self-driving car.

The true relative error is defined as:

$$
\epsilon_t = \frac{E_t}{\text{True}}
$$

or as a percentage:

$$
\epsilon_t (\%) = \frac{E_t}{\text{True}} \times 100\%
$$

### Example: What is the relative error from the previous calculation?

In [ ]:
eps_t = E_t / value_true
print(f"The true relative error is: {eps_t}")

## Approximate Absolute and Relative Error

What if we don't know the true value? Numerical methods often have a tunable parameter that controls accuracy (like $\Delta x$ above). We can estimate the error by comparing sequential approximations, using the *better* approximation in place of the true value.

$$
E_a = |\text{Better approximation} - \text{Approximation}|
$$

$$
\epsilon_a = \frac{E_a}{\text{Better approximation}}
$$

### Example: Use a smaller step size to find the approximate errors.

This is the typical use-case you will encounter in numerical methods since the true function is not usually known. 
> Except for *root finders* because we are looking for $x$ for which $f(x)=0$!

Commonly the difference is disregarded and the meaning has to be taken in context.

In [ ]:
approx_1 = approx_derivative(0.1)
approx_2 = approx_derivative(0.01)

E_a = np.abs(approx_2 - approx_1)
epsilon_a = E_a / approx_2

print(f"The approximate absolute error is: {E_a}")
print(f"The approximate relative error is: {epsilon_a}")

print(f"Recall, The true absolute and relative errors were: {E_t}, and {eps_t}")

## Tolerance

So how can we know when we have an answer that is *good enough*? We can't really (unless we know what the 'true' answer is!). Instead, the best we can do is compare our error indicators to a **tolerance**: 

- **Absolute Tolerance ($Tol_a$)**: The threshold below which the absolute error is considered acceptable.
- **Relative Tolerance ($Tol_r$)**: The threshold below which the relative error is considered acceptable.

> For approximate errors that are based on successive guesses, satisfying the tolerances doesn't *actually* indicate we've found the right solution, just that our esimate is *no longer improving*.

### Pseudocode Concept

```python
parameter = initial_value
while True:
    result = run_algorithm(parameter)
    error = calculate_error(result, previous_result)
    if error < tolerance:
        break
    parameter = reduce_parameter(parameter)
```

### Exploring $E_a$ and $\epsilon_a$ as a function of $\Delta x$

Lets see how the errors behave with decreasing step size. One might expect the error to approach zero as $\Delta x \rightarrow 0$. 

> Is this what you observe? What happens at $\Delta x = 10^{-8}$? Why? Checking the avlue of the function evaulations might shed some light on this!

In [ ]:
delta_x = np.logspace(0, -10, 11)
E_a = np.zeros(delta_x.size)
epsilon_a = np.zeros(delta_x.size)

for i, dx in enumerate(delta_x):
    # Note: We are comparing the approximation at dx with the one at dx/10
    E_a[i] = np.abs(approx_derivative(dx/10) - approx_derivative(dx))
    epsilon_a[i] = E_a[i] / approx_derivative(dx/10)

fig, ax = plt.subplots()
ax.loglog(delta_x, np.abs(E_a), marker='o', label='E_a')
ax.loglog(delta_x, np.abs(epsilon_a), marker='o', label='epsilon_a')
ax.set_xlabel('Delta x')
ax.set_ylabel('Error')
ax.legend()
plt.show()